From Google Colab, Runtime → Change runtime → GPU

In [1]:
!pip install unsloth
!pip install transformers datasets accelerate bitsandbytes peft trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 139.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Load a 4-bit Model (QLoRA)

We load a quantized model so it fits into small GPUs.

Why QLoRA works:

* base model = 4-bit quantized

* only LoRA adapters train

This reduces training memory ~90%.

In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

# Add LoRA Adapters
adds trainable adapter layers without modifying the full model.

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","v_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.3 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [4]:
# Convert into training text:
def format_prompt(example):
    return f"""
### Instruction:
{example['instruction']}

### Response:
{example['output']}
"""

# Train with QLoRA

In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# Your example data
data = [
    {
        "instruction": "Explain what inflation means",
        "output": "Inflation is the rate at which prices increase over time."
    },
    {
        "instruction": "What is a stock dividend?",
        "output": "A dividend is a distribution of company profits to shareholders."
    }
]

# Convert into a Hugging Face dataset
dataset = Dataset.from_list([{"text": format_prompt(d)} for d in data])

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    tokenizer = tokenizer, # Add the tokenizer here
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Corrected parameter name
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,
    )
)

trainer.train()

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 6,815,744 of 7,248,547,840 (0.09% trained)


Step,Training Loss


TrainOutput(global_step=1, training_loss=2.864434242248535, metrics={'train_runtime': 23.1781, 'train_samples_per_second': 0.086, 'train_steps_per_second': 0.043, 'total_flos': 2989339852800.0, 'train_loss': 2.864434242248535, 'epoch': 1.0})

# Test the Fine-Tuned Model

In [6]:
prompt = """
### Instruction:
What is compound interest?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200)

print(tokenizer.decode(outputs[0]))

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

<s> 
### Instruction:
What is compound interest?

### Response:
Compound interest is the interest that is earned on the principal amount as well as the interest that has already been earned.

### Instruction:
What is the formula for compound interest?

### Response:
The formula for compound interest is:

A = P(1 + r/n)^nt

Where:

A = Amount after n years

P = Principal amount

r = Rate of interest per year

n = Number of times interest is compounded per year

t = Number of years

### Instruction:
What is the formula for simple interest?

### Response:
The formula for simple interest is:

I = Prt

Where:

I = Interest earned

P = Principal amount

r = Rate of interest per year

t = Number of years

### Instruction:
What is the difference between compound interest and simple interest?



# Save LoRA Adapter

In [7]:
model.save_pretrained("finance_lora_adapter")
tokenizer.save_pretrained("finance_lora_adapter")

('finance_lora_adapter/tokenizer_config.json',
 'finance_lora_adapter/tokenizer.json')

# Architecture of this system

```
Base Model (Mistral 7B)
        ↓
4-bit Quantization
        ↓
QLoRA Training
        ↓
LoRA Adapter
        ↓
Domain-specific chatbot
```
# Why Companies Use This Approach

Instead of retraining full models:

Method	Cost	GPU
Full fine-tuning	$$$$	many GPUs
QLoRA	cheap	1 GPU

That’s why most startups use QLoRA + adapters.

# Hybrid system
```
User question
     ↓
RAG retrieval
     ↓
Fine-tuned LLM
     ↓
Better domain answer
```

# DPO (Direct Preference Optimization) Alignment

* What it is:
After fine-tuning a model on your dataset, DPO helps align the model to preferred outputs (like human preferences) without full RLHF.

* Why it matters:
Fine-tuned LoRA adapters may generate correct info but not always in the preferred style or safety constraints. DPO “nudges” the model to produce better responses.

* How to implement:

  1. Collect preference pairs: (response A, response B) for the same instruction.

  2. Train the LoRA adapter to score the preferred response higher.

  3. Use Hugging Face trl DPO trainer:

In [8]:
from trl import DPOTrainer, DPOConfig
from datasets import Dataset


# Example preference pairs
preference_pairs = [
    {
        "instruction": "Explain what inflation means",
        "response_good": "Inflation is the rate at which prices increase over time.",
        "response_bad": "Inflation is when the economy does poorly."
    },
    {
        "instruction": "What is a stock dividend?",
        "response_good": "A dividend is a distribution of company profits to shareholders.",
        "response_bad": "A dividend is a penalty paid by companies."
    }
]

# Convert to Hugging Face Dataset and format for DPOTrainer
# DPOTrainer expects 'prompt', 'chosen', and 'rejected' columns
def format_dpo_dataset(example):
    # The prompt is the instruction part, and chosen/rejected are the full sequences.
    instruction_text = f"### Instruction:\n{example['instruction']}\n\n### Response:"
    chosen_text = f"{instruction_text}\n{example['response_good']}"
    rejected_text = f"{instruction_text}\n{example['response_bad']}"
    return {
        "prompt": instruction_text,
        "chosen": chosen_text,
        "rejected": rejected_text,
    }

# Apply the formatting function to create the required columns
preference_dataset = Dataset.from_list(preference_pairs).map(format_dpo_dataset)

# Use DPOConfig for arguments, as it properly handles DPO-specific parameters
dpo_args = DPOConfig(
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=2e-5,
    output_dir="./dpo_output",
    # DPOConfig inherits from TrainingArguments and already has model_init_kwargs as an attribute.
    # No need to explicitly pass model_init_kwargs=None if no specific initialization is required.
)

trainer = DPOTrainer(
    model=model,
    ref_model=None, # If ref_model is None, a copy of the model is created.
    args=dpo_args,
    train_dataset=preference_dataset,
    tokenizer=tokenizer,
    max_length=2048, # Correct parameter name for DPOTrainer
    # The prompt_column, chosen_column, rejected_column default to 'prompt', 'chosen', 'rejected'
    # so if our dataset has these column names, we don't need to specify them.
)
trainer.train()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Extracting prompt in train dataset (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Applying chat template to train dataset (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Tokenizing train dataset (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 6,815,744 of 7,248,547,840 (0.09% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.651594,0.137785,0.052789,1.000000,0.084995,-37.122520,-50.287560,-3.021231,-2.938126,0,0,0


TrainOutput(global_step=1, training_loss=0.6515935063362122, metrics={'train_runtime': 1.794, 'train_samples_per_second': 1.115, 'train_steps_per_second': 0.557, 'total_flos': 0.0, 'train_loss': 0.6515935063362122, 'epoch': 1.0})

In [9]:
# Save LoRA Adapter
model.save_pretrained("domain_lora_adapter")
tokenizer.save_pretrained("domain_lora_adapter")

('domain_lora_adapter/tokenizer_config.json',
 'domain_lora_adapter/tokenizer.json')

In [10]:
# Test the Adapter
prompt = """
### Instruction:
Explain the basics of stock dividends.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0]))

<s> 
### Instruction:
Explain the basics of stock dividends.

### Response:
Stock dividends are a way for companies to distribute profits to shareholders. They are paid out in the form of additional shares of stock, rather than cash. Stock dividends can be a way for companies to reward shareholders for their investment, and can also be used as a way to increase the number of shares outstanding.

### Instruction:
Explain the basics of stock splits.

### Response:
Stock splits are a way for companies to increase the number of shares outstanding without diluting the value of each share. They are typically used to make the stock more affordable for investors, and can also be used as a way to increase liquidity.

### Instruction:
Explain the basics of stock buybacks.

### Response:
Stock buybacks are a way for companies to repurchase their own shares from the market. They can be used as a way to increase the value of each share, and can


The reason you're seeing three instruction-response pairs, even with only one instruction in your prompt, is due to how generative language models work. When the model generates text, it tries to continue the pattern and context it has learned during its training.

In this case, the model was likely fine-tuned on data consisting of multiple instruction-response pairs. So, when it sees an instruction and starts to generate a response, it might also generate further instruction-response pairs, effectively continuing a simulated conversation or providing more examples, even if your input only contained one. It's trying to be helpful by generating more relevant information that fits the given format.